In [3]:
import math
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

# Constants
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    """Build image transformation pipeline."""
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    """Find the closest aspect ratio from target ratios."""
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    """Dynamically preprocess image into tiles."""
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    
    # Calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) 
        for i in range(1, n + 1) 
        for j in range(1, n + 1) 
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    
    # Find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    
    # Calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    
    # Resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # Split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    
    assert len(processed_images) == blocks
    
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    """Load and preprocess a single image."""
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# Model path
path = "OpenGVLab/InternVL3-14B"

print("Loading model and tokenizer...")
# Load model (16-bit bf16)
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)
print("Model loaded successfully!")


Loading model and tokenizer...


KeyboardInterrupt: 

In [ ]:
# Load two images
print("\nLoading images...")
image1_path = '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_hands/A chef preparing gou/seed_2/layer_3.png'  # Replace with your first image path
image2_path = '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_hands/A chef preparing gou/seed_2/layer_14.png'  # Replace with your second image path

pixel_values1 = load_image(image1_path, max_num=12).to(torch.bfloat16).cuda()
pixel_values2 = load_image(image2_path, max_num=12).to(torch.bfloat16).cuda()

# Concatenate images
pixel_values = torch.cat((pixel_values1, pixel_values2), dim=0)
num_patches_list = [pixel_values1.size(0), pixel_values2.size(0)]

print(f"Image 1 patches: {pixel_values1.size(0)}")
print(f"Image 2 patches: {pixel_values2.size(0)}")

# Generation configuration
generation_config = dict(max_new_tokens=1024, do_sample=True)

# Method 1: Combined images (images are concatenated)
print("\n" + "="*80)
print("Method 1: Hand Anatomy Quality Assessment")
print("="*80)
question = '''<image>
Both images were generated with the prompt: "A chef preparing gourmet food in a modern kitchen, human hands visible chopping vegetables"

Please analyze the hand anatomy in both images carefully. For each image, evaluate:
1. Number of fingers (should be 5 per hand)
2. Finger proportions and positioning (natural vs. distorted)
3. Thumb placement and structure
4. Joint articulation (knuckles, finger bends)
5. Overall hand shape and palm structure
6. Hand size relative to objects being held
7. Any anatomical abnormalities (extra/missing fingers, merged digits, impossible angles)

Based on this analysis, which image has MORE ACCURATE hand anatomy? Provide specific details about the hand errors you observe in each image, then clearly state which image is better.'''

response, history = model.chat(
    tokenizer, 
    pixel_values, 
    question, 
    generation_config,
    history=None, 
    return_history=True
)
print(f'User: {question}')
print(f'Assistant: {response}\n')


Loading images...
Image 1 patches: 10
Image 2 patches: 10

Method 1: Hand Anatomy Quality Assessment


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Both images were generated with the prompt: "A chef preparing gourmet food in a modern kitchen, human hands visible chopping vegetables"

Please analyze the hand anatomy in both images carefully. For each image, evaluate:
1. Number of fingers (should be 5 per hand)
2. Finger proportions and positioning (natural vs. distorted)
3. Thumb placement and structure
4. Joint articulation (knuckles, finger bends)
5. Overall hand shape and palm structure
6. Hand size relative to objects being held
7. Any anatomical abnormalities (extra/missing fingers, merged digits, impossible angles)

Based on this analysis, which image has MORE ACCURATE hand anatomy? Provide specific details about the hand errors you observe in each image, then clearly state which image is better.
Assistant: Let's analyze the images based on the specified criteria:

### Image 1 Analysis:
1. **Number of Fingers**: The number of fingers appears correct with five per hand.
2. **Finger Proportions and Positioning**:

In [ ]:
# Load single image
print("\nLoading image...")
image_path = '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_hands/A chef preparing gou/seed_2/layer_10.png'

pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).cuda()

print(f"Image patches: {pixel_values.size(0)}")

# Generation configuration
generation_config = dict(max_new_tokens=1024, do_sample=True)

# Single image hand anatomy evaluation with scoring
print("\n" + "="*80)
print("Hand Anatomy Quality Assessment")
print("="*80)
question = '''<image>
This image was generated with the prompt: "A chef preparing gourmet food in a modern kitchen, human hands visible chopping vegetables"

Please analyze the hand anatomy in this image carefully. Evaluate:
1. Number of fingers (should be 5 per hand)
2. Finger proportions and positioning (natural vs. distorted)
3. Thumb placement and structure
4. Joint articulation (knuckles, finger bends)
5. Overall hand shape and palm structure
6. Hand size relative to objects being held
7. Any anatomical abnormalities (extra/missing fingers, merged digits, impossible angles)

Provide a detailed analysis of any hand errors you observe, then assign a HAND QUALITY SCORE from 0-10, where:
- 10 = Perfect, anatomically correct hands
- 7-9 = Minor issues (slight distortions, unclear details)
- 4-6 = Moderate issues (wrong proportions, ambiguous finger count)
- 1-3 = Major issues (extra/missing fingers, severe distortions)
- 0 = Completely malformed or no visible hands

End your response with: "SCORE: X/10"'''

response, history = model.chat(
    tokenizer, 
    pixel_values, 
    question, 
    generation_config,
    history=None, 
    return_history=True
)
print(f'User: {question}')
print(f'Assistant: {response}\n')

# Extract score from response (optional)
import re
score_match = re.search(r'SCORE:\s*(\d+)/10', response)
if score_match:
    score = int(score_match.group(1))
    print(f"\nExtracted Score: {score}/10")
else:
    print("\nWarning: Could not extract score from response")


Loading image...
Image patches: 10

Hand Anatomy Quality Assessment


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
This image was generated with the prompt: "A chef preparing gourmet food in a modern kitchen, human hands visible chopping vegetables"

Please analyze the hand anatomy in this image carefully. Evaluate:
1. Number of fingers (should be 5 per hand)
2. Finger proportions and positioning (natural vs. distorted)
3. Thumb placement and structure
4. Joint articulation (knuckles, finger bends)
5. Overall hand shape and palm structure
6. Hand size relative to objects being held
7. Any anatomical abnormalities (extra/missing fingers, merged digits, impossible angles)

Provide a detailed analysis of any hand errors you observe, then assign a HAND QUALITY SCORE from 0-10, where:
- 10 = Perfect, anatomically correct hands
- 7-9 = Minor issues (slight distortions, unclear details)
- 4-6 = Moderate issues (wrong proportions, ambiguous finger count)
- 1-3 = Major issues (extra/missing fingers, severe distortions)
- 0 = Completely malformed or no visible hands

End your response with: "SC

In [ ]:
def evaluate_text_quality_two_step(image_path, generation_prompt, tokenizer, model, load_image):
    """
    Evaluate text rendering quality using a two-step process:
    1. First get detailed description of all visible text
    2. Then score based on that description
    
    Args:
        image_path: Path to the image file
        expected_texts: List of text strings that should appear
        generation_prompt: The prompt used to generate the image
        tokenizer: Model tokenizer
        model: Vision-language model
        load_image: Image loading function
    
    Returns:
        tuple: (score, description, scoring_response)
    """
    # Load image
    pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).cuda()
    
    # Generation configuration
    generation_config = dict(max_new_tokens=1024, do_sample=True)
    
    # STEP 1: Get detailed description of all text
    print("\n" + "="*80)
    print("STEP 1: Text Description")
    print("="*80)
    
    question1 = '''<image>
CRITICAL TASK: Describe every single bit of text that is visible in the image in detail. List all text exactly as it appears, character by character.'''
    
    description, history = model.chat(
        tokenizer, 
        pixel_values, 
        question1, 
        generation_config,
        history=None, 
        return_history=True
    )
    
    print(f'User: {question1}')
    print(f'Assistant: {description}\n')
    
    # STEP 2: Score based on the description
    print("\n" + "="*80)
    print("STEP 2: Text Quality Scoring")
    print("="*80)
        
    question2 = f'''Based on your previous description of the text in the image, now evaluate the text quality.

This image was generated with: "{generation_prompt}"

Compare your description of visible text versus what text was expected. Check for:
- Wrong letters
- Missing or extra letters
- Completely hallucinated text that shouldn't be there
- Spelling errors
- Any other text defects

Use STRICT scoring criteria.
End with: "SCORE: X/10"'''
    
    scoring_response, _ = model.chat(
        tokenizer, 
        pixel_values, 
        question2, 
        generation_config,
        history=history,  # Use the history from step 1
        return_history=True
    )
    
    print(f'User: {question2}')
    print(f'Assistant: {scoring_response}\n')
    
    # Extract score
    import re
    score_match = re.search(r'SCORE:\s*(\d+)/10', scoring_response)
    score = int(score_match.group(1)) if score_match else None
    
    if score is not None:
        print(f"\nExtracted Score: {score}/10")
        if score < 10:
            print(f"⚠️  TEXT DEFECTS DETECTED")
        else:
            print(f"✓ Perfect text rendering")
    else:
        print("\nWarning: Could not extract score")
    
    return score, description, scoring_response


# USAGE:
score, description, scoring = evaluate_text_quality_two_step(
    image_path='/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_text/A fantasy bakery run/seed_4/layer_4.png',
    generation_prompt="A fantasy bakery run by an orc. The window sign says 'SWEET TOOTH'S DELIGHT'. Cupcakes are labeled 'Lava Fudge' and 'Goblin Berry Swirl'. The orc’s apron has an embroidered phrase: 'BAKE OR BREAK'.",
    tokenizer=tokenizer,
    model=model,
    load_image=load_image
)

print(f"\nFinal Score: {score}/10")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



STEP 1: Text Description


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
CRITICAL TASK: Describe every single bit of text that is visible in the image in detail. List all text exactly as it appears, character by character.
Assistant: The text visible in the image includes:

1. On the lantern: 
   - "Tooth's Delight"

2. On the apron:
   - "BAKE OR BREAK"

3. On the paper in front:
   - "Lava Fudge Goblin Berry Swirl"


STEP 2: Text Quality Scoring
User: Based on your previous description of the text in the image, now evaluate the text quality.

This image was generated with: "A fantasy bakery run by an orc. The window sign says 'SWEET TOOTH'S DELIGHT'. Cupcakes are labeled 'Lava Fudge' and 'Goblin Berry Swirl'. The orc’s apron has an embroidered phrase: 'BAKE OR BREAK'."

Compare your description of visible text versus what text was expected. Check for:
- Wrong letters
- Missing or extra letters
- Completely hallucinated text that shouldn't be there
- Spelling errors
- Any other text defects

Use STRICT scoring criteria.
End with: "SCORE: X/10

In [ ]:
def evaluate_text_quality_two_step(image_path, generation_prompt, tokenizer, model, load_image):
    """
    Evaluate text rendering quality using a two-step process:
    1. First get detailed description of all visible text
    2. Then score based on that description
    
    Args:
        image_path: Path to the image file
        expected_texts: List of text strings that should appear
        generation_prompt: The prompt used to generate the image
        tokenizer: Model tokenizer
        model: Vision-language model
        load_image: Image loading function
    
    Returns:
        tuple: (score, description, scoring_response)
    """
    # Load image
    pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).cuda()
    
    # Generation configuration
    generation_config = dict(max_new_tokens=1024, do_sample=True)
    
    # STEP 1: Get detailed description of all text
    print("\n" + "="*80)
    print("STEP 1: Text Description")
    print("="*80)
    
    question1 = '''<image>
CRITICAL TASK: Give a detailed description of the image. Then list all things that could possibly increase or decrease the aesthetic value of the image if assessed by a human and specify how relevant each point is in detail.'''
    
    description, history = model.chat(
        tokenizer, 
        pixel_values, 
        question1, 
        generation_config,
        history=None, 
        return_history=True
    )
    
    print(f'User: {question1}')
    print(f'Assistant: {description}\n')
    
    # STEP 2: Score based on the description
    print("\n" + "="*80)
    print("STEP 2: Text Quality Scoring")
    print("="*80)
        
    question2 = f'''Based on your previous description of the image's aesthetic qualities, now provide an overall aesthetic score.

For reference, this image was generated with: "{generation_prompt}"

Use VERY STRICT scoring critera.
End with: "SCORE: X/10"'''
    
    scoring_response, _ = model.chat(
        tokenizer, 
        pixel_values, 
        question2, 
        generation_config,
        history=history,  # Use the history from step 1
        return_history=True
    )
    
    print(f'User: {question2}')
    print(f'Assistant: {scoring_response}\n')
    
    # Extract score
    import re
    score_match = re.search(r'SCORE:\s*(\d+)/10', scoring_response)
    score = int(score_match.group(1)) if score_match else None
    
    if score is not None:
        print(f"\nExtracted Score: {score}/10")
        if score < 10:
            print(f"⚠️  TEXT DEFECTS DETECTED")
        else:
            print(f"✓ Perfect text rendering")
    else:
        print("\nWarning: Could not extract score")
    
    return score, description, scoring_response


# USAGE:
score, description, scoring = evaluate_text_quality_two_step(
    image_path='/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_aesthetics/a moody city street /seed_2/layer_12.png',
    generation_prompt="a moody city street beneath a dramatic, swirling cloudscape, captured with rich contrast and cinematic lighting",
    tokenizer=tokenizer,
    model=model,
    load_image=load_image
)

print(f"\nFinal Score: {score}/10")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



STEP 1: Text Description


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
CRITICAL TASK: Give a detailed description of the image. Then list all things that could possibly increase or decrease the aesthetic value of the image if assessed by a human and specify how relevant each point is in detail.
Assistant: The image depicts a quiet urban street scene under a dramatic, cloudy sky. The street is empty, bordered by buildings on both sides with closed shopfronts. Street lamps cast a soft glow, creating a moody and atmospheric feel, enhanced by the impending stormlight filtering through the thick clouds. The wet pavement reflects the ambient light, adding depth and highlighting the deserted nature of the street.

### Elements that Could Increase Aesthetic Value:

1. **Lighting:**
   - **Relevance:** High. The interplay of light and shadow is critical in setting the mood. Consider adjusting the intensity or adding additional light sources for more contrast or highlight specific parts, enhancing the dramatic effect.

2. **Color Contrast:**
   - **Re

In [ ]:
import math
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import gc

def flush():
    gc.collect()
    torch.cuda.empty_cache()

def build_transform(input_size):
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

def get_evaluation_prompts(target):
    """
    Returns target-specific evaluation prompts.
    
    Args:
        target: The evaluation target (e.g., 'text', 'hands', 'aesthetics')
    
    Returns:
        tuple: (description_prompt, scoring_prompt_template)
    """
    prompts = {
        'text': {
            'description': '''<image>
            CRITICAL TASK: Describe every single bit of text that is visible in the image in detail. List all text exactly as it appears, character by character.''',

            'scoring': '''The image was generated with this prompt: "{generation_prompt}"
            Based on your previous description of the text in the image, now evaluate the quality of the generated text in the image.
    
            Compare your description of visible text versus what text was expected. Check for:
            - Wrong letters
            - Missing or extra letters
            - Completely hallucinated text that shouldn't be there
            - Spelling errors
            - Any other text defects
            - ...

            Use VERY STRICT scoring criteria.
            End with: "SCORE: X/10"'''
        },

        'hands': {
            'description': '''<image>
            CRITICAL TASK: Describe every single human hand visible in the image in great detail. For each hand, describe:
            - Number of fingers
            - Position and pose of the hand
            - Any anatomical issues or abnormalities
            - Whether fingers look natural and properly formed
            - Connection to arms/wrists,
            - ...''',

            'scoring': '''
            Based on your previous description of the hands in the image, now critically evaluate the quality of the hands in the image.

            Evaluate based on:
            - General visibility of hands
            - Correct number of fingers (should be 5 per hand if hand is fully visible)
            - Natural finger proportions and positions
            - Proper hand anatomy
            - Realistic joints and connections
            - No extra or missing fingers
            - No distorted or malformed fingers
            - Natural hand poses
            - ...

            Use VERY STRICT scoring criteria.
            End with: "SCORE: X/10"'''
        },

        'aesthetics': {
            'description': '''<image>
            CRITICAL TASK: Describe the image in great detail. In your description, include all properties of the image that increase or decrease its aesthetic value and their relevance for its overall aesthetic value. Be very critical. Include:
            - Overall image quality 
            - Composition and framing
            - Color harmony and palette
            - Lighting quality
            - Visual appeal and artistic merit
            - Technical quality (sharpness, exposure, etc.)
            - Overall mood and atmosphere
            - ...''',
            'scoring': '''
            Based on your previous description of the image's aesthetic qualities, now provide an overall aesthetic score. 
            Make sure that images that are beautiful, but not outstandingly aesthetic, do not get high scores. 

            Use VERY STRICT scoring criteria. 
            End with: "SCORE: X/10"'''
        }
    }
    
    if target not in prompts:
        raise ValueError(f"Unknown target: {target}. Available targets: {list(prompts.keys())}")
    
    return prompts[target]['description'], prompts[target]['scoring']

def evaluate_quality_two_step(image_path, generation_prompt, target, tokenizer, model, load_image):
    """
    Evaluate image quality using a two-step process:
    1. First get detailed description of the target aspect
    2. Then score based on that description
    
    Args:
        image_path: Path to the image file
        generation_prompt: The prompt used to generate the image
        target: What to evaluate (e.g., 'text', 'hands', 'aesthetics')
        tokenizer: Model tokenizer
        model: Vision-language model
        load_image: Image loading function
    
    Returns:
        tuple: (score, description, scoring_response)
    """
    description_prompt, scoring_template = get_evaluation_prompts(target)
    
    pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).cuda()
    
    generation_config = dict(max_new_tokens=1024, do_sample=True)
    
    # STEP 1: Get detailed description
    description, history = model.chat(
        tokenizer, 
        pixel_values, 
        description_prompt, 
        generation_config,
        history=None, 
        return_history=True
    )
    
    # STEP 2: Score based on the description
    scoring_prompt = scoring_template.format(generation_prompt=generation_prompt)
    
    scoring_response, _ = model.chat(
        tokenizer, 
        pixel_values, 
        scoring_prompt, 
        generation_config,
        history=history,
        return_history=True
    )

    print(scoring_response)
    
    # Extract score - handles both integer and decimal scores
    last_line = scoring_response.strip().split('\n')[-1]
    score_match = re.search(r'SCORE:\s*(\d+(?:\.\d+)?)/10', last_line, re.IGNORECASE)    score = float(score_match.group(1)) if score_match else None
    
    return score, description, scoring_response

def create_prompt_mapping(prompts_list):
    """Create mapping from directory name (first 20 chars) to full prompt"""
    mapping = {}
    for prompt in prompts_list:
        # Handle both string prompts and dict prompts
        if isinstance(prompt, str):
            full_prompt = prompt
        elif isinstance(prompt, dict):
            full_prompt = prompt.get('positive', prompt.get('prompt', ''))
        else:
            full_prompt = str(prompt)
        
        # Create directory name (first 20 chars)
        dir_name = full_prompt[:20]
        mapping[dir_name] = full_prompt
    
    return mapping

SyntaxError: invalid syntax (616867325.py, line 213)

In [ ]:
import json
import os 


with open('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/prompt_datasets/aesthetics/aesthetics.json', 'r') as f:
    prompts_list = json.load(f)

# Create mapping from directory name to full prompt
prompt_mapping = create_prompt_mapping(prompts_list)
print(f"Loaded {len(prompt_mapping)} prompts")

# Load InternVL3 Model
print("Loading InternVL3 model...")
model_path = "OpenGVLab/InternVL3-14B"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True
).eval().cuda()

quality_results = []

description_prompt, scoring_template = get_evaluation_prompts('aesthetics')
print('Using description prompt:')
print(description_prompt)
print('Using evaluation prompt:')
print(scoring_template)

# Get all prompt directories
prompt_dirs = [d for d in os.listdir('/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_aesthetics') 
                if os.path.isdir(os.path.join('/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_aesthetics', d))]

print(f"\nFound {len(prompt_dirs)} prompt directories")

for prompt_dir in prompt_dirs[:2]:
    prompt_path = os.path.join('/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_aesthetics', prompt_dir)
    
    # Get full prompt from mapping
    if prompt_dir not in prompt_mapping:
        print(f"Warning: No prompt found for directory '{prompt_dir}', skipping...")
        continue
    
    prompt = prompt_mapping[prompt_dir]
    
    print(f"\n{'='*80}")
    print(f"Processing prompt: {prompt}")
    print(f"{'='*80}")
    
    prompt_results = {
        'prompt': prompt,
        'prompt_dir': prompt_dir,
        'target': 'aesthetics',
        'seeds': []
    }
    
    # Get all seed directories
    seed_dirs = [d for d in os.listdir(prompt_path) 
                    if os.path.isdir(os.path.join(prompt_path, d)) and d.startswith('seed_')]
    
    for seed_dir in seed_dirs[:2]:
        seed_path = os.path.join(prompt_path, seed_dir)
        
        # Extract seed number
        seed_num = seed_dir.replace('seed_', '')
        
        seed_result = {
            'seed': seed_num,
            'layers': []
        }
        
        # Process each layer image
        for layer_idx in range(2):
            image_path = os.path.join(seed_path, f'layer_{layer_idx}.png')
            
            if not os.path.exists(image_path):
                print(f"Warning: Image not found: {image_path}")
                continue
            
            print(f"Processing layer {layer_idx}...")
            
            # Evaluate quality with retry until valid score
            max_retries = 10
            retry_count = 0
            score = None
            
            while score is None and retry_count < max_retries:
                try:
                    score, description, scoring_response = evaluate_quality_two_step(
                        image_path=image_path,
                        generation_prompt=prompt,
                        target='aesthetics',
                        tokenizer=tokenizer,
                        model=model,
                        load_image=load_image
                    )
                    
                    if score is not None:
                        print(score)
                        layer_result = {
                            'layer': layer_idx,
                            'score': score,
                            'description': description,
                            'scoring_response': scoring_response,
                            'retry_count': retry_count,
                            'image_path': image_path
                        }
                        seed_result['layers'].append(layer_result)
                        print(f"Layer {layer_idx} Score: {score}/10 (attempts: {retry_count + 1})")
                    else:
                        retry_count += 1
                        print(f"No score extracted, retrying... (attempt {retry_count}/{max_retries})")
                    
                except Exception as e:
                    retry_count += 1
                    print(f"Error evaluating layer {layer_idx} (attempt {retry_count}/{max_retries}): {str(e)}")
                    if retry_count >= max_retries:
                        layer_result = {
                            'layer': layer_idx,
                            'score': None,
                            'error': str(e),
                            'retry_count': retry_count,
                            'image_path': image_path
                        }
                        seed_result['layers'].append(layer_result)
                        print(f"Max retries reached for layer {layer_idx}")
            
            if score is None and retry_count >= max_retries:
                print(f"Failed to get valid score for layer {layer_idx} after {max_retries} attempts")
            
            flush()
        
        # Calculate average score for this seed
        valid_scores = [l['score'] for l in seed_result['layers'] if l['score'] is not None]
        seed_result['avg_score'] = sum(valid_scores) / len(valid_scores) if valid_scores else None
        
        prompt_results['seeds'].append(seed_result)
        
        if seed_result['avg_score'] is not None:
            print(f"Average score for {seed_dir}: {seed_result['avg_score']:.2f}/10")
    
    # Calculate overall statistics across all seeds
    all_scores = [s['avg_score'] for s in prompt_results['seeds'] if s['avg_score'] is not None]
    prompt_results['overall_avg_score'] = sum(all_scores) / len(all_scores) if all_scores else None
    
    quality_results.append(prompt_results)
    
    if prompt_results['overall_avg_score'] is not None:
        print(f"\nOverall average score for this prompt: {prompt_results['overall_avg_score']:.2f}/10")
    
    # Save intermediate results
    output_path = f'aesthetics_quality_scores.json'
    with open(output_path, 'w') as f:
        json.dump(quality_results, f, indent=2)
    
    flush()

print(f"\n{'='*80}")
print(f"Evaluation complete! Results saved to {output_path}")
print(f"{'='*80}")

Loaded 20 prompts
Loading InternVL3 model...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Using description prompt:
<image>
            CRITICAL TASK: Describe the image in great detail. In your description, include all properties of the image that increase or decrease its aesthetic value and their relevance for its overall aesthetic value. Be very critical. Include:
            - Overall image quality 
            - Composition and framing
            - Color harmony and palette
            - Lighting quality
            - Visual appeal and artistic merit
            - Technical quality (sharpness, exposure, etc.)
            - Overall mood and atmosphere
            - ...
Using evaluation prompt:

            Based on your previous description of the image's aesthetic qualities, now provide an overall aesthetic score. 
            Make sure that images that are beautiful, but not outstandingly aesthetic, do not get high scores. 

            Use VERY STRICT scoring criteria. 
            End with: "SCORE: X/10"

Found 20 prompt directories

Processing prompt: a serene sce

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


The image has several elements that contribute positively to its aesthetic value, such as a well-balanced composition, vibrant color palette, and serene mood. However, it suffers from technical issues like pixelation, noise, and washed-out highlights that significantly detract from its visual clarity and overall impact.

**Color Harmony and Palette:** 7/10

**Composition and Framing:** 7/10

**Lighting Quality:** 6/10

**Visual Appeal and Artistic Merit:** 6/10

**Technical Quality (Sharpness, Exposure, etc.):** 3/10

**Overall Mood and Atmosphere:** 7/10

Considering these factors collectively, the image has potential but is let down by its technical limitations.

**Overall Aesthetic Score:**
SCORE: 6/10
6.0
Layer 0 Score: 6.0/10 (attempts: 1)
Processing layer 1...


...

### Aesthetic Score Breakdown:
- **Visual Appeal and Artistic Merit**: 6/10  
  The use of glitch art is effective and appealing to niche audiences, creating a distinct visual experience.
  
- **Color Harmony and Palette**: 5/10  
  The vibrant colors are eye-catching but lack subtlety, leading to a somewhat overwhelming effect.

- **Composition and Framing**: 5/10  
  Simple but recognizable objects are framed well enough to draw attention, though the distortion sacrifices traditional composition.

- **Technical Quality**: 4/10  
  The intentional low sharpness and exposure flaws, while part of the style, detract from clarity.

- **Mood and Atmosphere**: 6/10  
  The mood balances nostalgia and chaos, though this can be ambiguous and uneven.

### Overall Aesthetic Score
Balancing these aspects and considering the strict criteria, the image effectively harnesses a unique visual style but does not reach exceptional aesthetic heights due to technical flaws and potential audience polariza

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


To evaluate the overall aesthetic score of the image, let’s consider the key aesthetic components discussed, factoring in the strict criteria mentioned:

### Evaluation Breakdown:

1. **Overall Image Quality**: The pixelation and color distortion add an artistic touch but reduce realism. **Score: 6/10**
   
2. **Composition and Framing**: The balanced composition and use of symmetry effectively draw attention. **Score: 7/10**

3. **Color Harmony and Palette**: While the muted tones are appealing, the color fringes introduce artificiality. **Score: 5/10**

4. **Lighting Quality**: The warm, focused light effectively enhances the mood. **Score: 8/10**

5. **Visual Appeal and Artistic Merit**: The artistic choice of style is creative but limited in broader appeal. **Score: 6/10**

6. **Technical Quality**: The quality is intentionally altered, which can be a strength but compromises sharpness. **Score: 5/10**

7. **Overall Mood and Atmosphere**: The nostalgic and cozy feel is strong and e

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Based on the described aesthetic qualities, the image demonstrates a unique charm and nostalgia, though it falls short in certain technical aspects and color vibrancy. Here's the analysis leading to the score:

- **Composition and Framing (8/10):** Simple and balanced, keeping focus on key elements.
- **Color Harmony and Palette (7/10):** Vibrant but slightly muted due to pixelation.
- **Lighting Quality (8/10):** Soft and inviting, contributing to the warm atmosphere.
- **Visual Appeal and Artistic Merit (6/10):** Engaging due to retro style, but pixelation detracts slightly.
- **Technical Quality (6/10):** Moderately impacted by intentional pixelation.
- **Overall Mood and Atmosphere (8/10):** Whimsical and nostalgic, very cohesive.

**Overall Aesthetic Score:**
Considering the balance of artistic merits and technical limitations, the image scores as follows:

SCORE: 7/10
7.0
Layer 1 Score: 7.0/10 (attempts: 1)
Average score for seed_2: 6.50/10

Overall average score for this prompt:

In [ ]:
json.load()

AttributeError: 'str' object has no attribute 'read'

In [ ]:
import json
with open('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/text_quality_scores.json', 'r') as f:
        prompts_list = json.load(f)

In [ ]:
len(prompts_list)

10

In [ ]:
prompts_list

[{'prompt': "A steam-powered mech piloted by a mouse. On the mech's chestplate is etched 'THE BRASS CLAW'. Gauges display labels like 'PRESSURE MAX' and 'OIL LEVEL LOW'. The cockpit has sticky notes with scribbled text.",
  'prompt_dir': 'A steam-powered mech',
  'target': 'text',
  'seeds': [{'seed': '2',
    'layers': [{'layer': 0,
      'score': 6.0,
      'image_path': '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_text/A steam-powered mech/seed_2/layer_0.png'},
     {'layer': 1,
      'score': 2.0,
      'image_path': '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_text/A steam-powered mech/seed_2/layer_1.png'},
     {'layer': 2,
      'score': 4.0,
      'image_path': '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_text/A steam-powered mech/seed_2/layer_2.png'},
     {'layer': 3,
      'score': 8.0,
      'image_path': '/export/scratch/ru63zus/msg_images/experiments/flux_layer_ablation_text/A steam-powered mech/seed_2/l

In [ ]:
import json

# Assuming your JSON data is in a file called 'data.json'
# If you have it as a string or already loaded, adjust accordingly
with open('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/aesthetics_quality_scores.json', 'r') as f:
    data = json.load(f)

# Initialize a dictionary to store scores for each layer
layer_scores = {}

# Iterate through all prompts
for prompt_data in data:
    # Iterate through all seeds
    for seed_data in prompt_data['seeds']:
        # Iterate through all layers
        for layer_data in seed_data['layers']:
            layer_num = layer_data['layer']
            score = layer_data['score']
            
            # Add score to the appropriate layer
            if layer_num not in layer_scores:
                layer_scores[layer_num] = []
            layer_scores[layer_num].append(score)

# Calculate average for each layer
layer_averages = {}
for layer_num, scores in sorted(layer_scores.items()):
    avg_score = sum(scores) / len(scores)
    layer_averages[layer_num] = avg_score
    print(f"Layer {layer_num}: Average Score = {avg_score:.4f} (n={len(scores)})")

# Optional: Create a summary
print(f"\n{'='*50}")
print("Summary:")
print(f"{'='*50}")
for layer_num, avg_score in layer_averages.items():
    print(f"Layer {layer_num:2d}: {avg_score:.4f}")

Layer 0: Average Score = 6.9640 (n=50)
Layer 1: Average Score = 6.4220 (n=50)
Layer 2: Average Score = 4.7220 (n=50)
Layer 3: Average Score = 8.7210 (n=50)
Layer 4: Average Score = 8.7840 (n=50)
Layer 5: Average Score = 8.8160 (n=50)
Layer 6: Average Score = 8.9060 (n=50)
Layer 7: Average Score = 8.8876 (n=50)
Layer 8: Average Score = 8.7370 (n=50)
Layer 9: Average Score = 8.6680 (n=50)
Layer 10: Average Score = 8.4680 (n=50)
Layer 11: Average Score = 8.6560 (n=50)
Layer 12: Average Score = 8.6980 (n=50)
Layer 13: Average Score = 8.7340 (n=50)
Layer 14: Average Score = 8.7890 (n=50)
Layer 15: Average Score = 9.0680 (n=50)
Layer 16: Average Score = 8.8600 (n=50)
Layer 17: Average Score = 8.9640 (n=50)
Layer 18: Average Score = 8.2620 (n=50)

Summary:
Layer  0: 6.9640
Layer  1: 6.4220
Layer  2: 4.7220
Layer  3: 8.7210
Layer  4: 8.7840
Layer  5: 8.8160
Layer  6: 8.9060
Layer  7: 8.8876
Layer  8: 8.7370
Layer  9: 8.6680
Layer 10: 8.4680
Layer 11: 8.6560
Layer 12: 8.6980
Layer 13: 8.7340
La

In [ ]:
vlm_text_mean = torch.tensor([5.8700, 4.9300, 3.0600, 5.8100, 5.4200, 5.4900, 5.8700, 5.2800, 5.6800, 5.5100, 5.5400, 5.4600, 5.9400, 5.5600, 5.7800, 5.5000, 5.7100,6.1300, 6.7800])
ocr_text = torch.tensor([0.231960, 0.230319, 0.147445, 0.211755, 0.210885, 0.202199, 0.216633, 0.206342, 0.222167, 0.195722, 0.210882, 0.198651, 0.204099, 0.205225, 0.216540, 0.215737, 0.215030, 0.216897, 0.216760])

In [6]:
import numpy as np
# Pearson correlation
corr_matrix = torch.corrcoef(torch.stack([vlm_text, ocr_text]))
pearson = corr_matrix[0, 1].item()

print(f"Pearson Correlation: {pearson:.6f}")

# Also with numpy for verification
np_pearson = np.corrcoef(vlm_text.numpy(), ocr_text.numpy())[0, 1]
print(f"Pearson Correlation (numpy): {np_pearson:.6f}")

# Spearman correlation
from scipy.stats import spearmanr
spearman, p_value = spearmanr(vlm_text.numpy(), ocr_text.numpy())
print(f"Spearman Correlation: {spearman:.6f} (p-value: {p_value:.6f})")

Pearson Correlation: 0.447255
Pearson Correlation (numpy): 0.447255
Spearman Correlation: 0.564706 (p-value: 0.022663)


In [7]:
vlm_hands = torch.tensor([9.6850,5.4190,6.0820,10.9640,11.0550,9.5725,9.5080,9.5400,11.2960,9.6700,9.5100,9.7030,9.6820,9.5780,10.9300,9.4660,9.4650,12.3530,9.7900][3:])
classifier_hands = torch.tensor([0.186552, 0.136625, 0.133595,  0.709330,  0.666218,  0.697655,   0.650028,  0.662389,   0.652602, 0.599139,  0.751199,  0.585819,   0.694342,   0.697953,  0.693330,   0.661349,   0.639252,    0.645254,   0.651743][3:])

In [8]:

# Pearson correlation
corr_matrix = torch.corrcoef(torch.stack([vlm_hands, classifier_hands]))
pearson = corr_matrix[0, 1].item()

print(f"Pearson Correlation: {pearson:.6f}")

# Also with numpy for verification
np_pearson = np.corrcoef(vlm_hands.numpy(), classifier_hands.numpy())[0, 1]
print(f"Pearson Correlation (numpy): {np_pearson:.6f}")

# Spearman correlation
from scipy.stats import spearmanr
spearman, p_value = spearmanr(vlm_hands.numpy(), classifier_hands.numpy())
print(f"Spearman Correlation: {spearman:.6f} (p-value: {p_value:.6f})")

Pearson Correlation: -0.002850
Pearson Correlation (numpy): -0.002850
Spearman Correlation: -0.002941 (p-value: 0.991375)


In [9]:
vlm_aesthetics = torch.tensor([6.9640, 6.4220, 4.7220, 8.7210,8.7840, 8.8160, 8.9060,8.8876,8.7370,8.6680, 8.4680,8.6560,8.6980,8.7340,8.7890,9.0680,8.8600, 8.9640,8.2620][3:])
classifier_aesthetics = torch.tensor([6.414817, 6.012494,  5.183740,     6.884494,     6.954563,      6.978050,      7.020353,      6.937215,      7.028258,      6.980995,     7.022676,     7.007033,     7.024124,     7.002501,     7.043288,     7.050234,     6.891521,     6.890457,     6.825743][3:])

In [10]:
# Pearson correlation
corr_matrix = torch.corrcoef(torch.stack([vlm_aesthetics, classifier_aesthetics]))
pearson = corr_matrix[0, 1].item()

print(f"Pearson Correlation: {pearson:.6f}")

# Also with numpy for verification
np_pearson = np.corrcoef(vlm_aesthetics.numpy(), classifier_aesthetics.numpy())[0, 1]
print(f"Pearson Correlation (numpy): {np_pearson:.6f}")

# Spearman correlation
from scipy.stats import spearmanr
spearman, p_value = spearmanr(vlm_aesthetics.numpy(), classifier_aesthetics.numpy())
print(f"Spearman Correlation: {spearman:.6f} (p-value: {p_value:.6f})")

Pearson Correlation: 0.288485
Pearson Correlation (numpy): 0.288485
Spearman Correlation: 0.094118 (p-value: 0.728814)


In [2]:
import json
import torch
import numpy as np
from scipy.stats import spearmanr, pearsonr

# Load data
with open('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/text_quality_scores.json', 'r') as f:
    data = json.load(f)

# Baseline performance per layer (your OCR scores)
baseline = torch.tensor([0.231960, 0.230319, 0.147445, 0.211755, 0.210885, 
                         0.202199, 0.216633, 0.206342, 0.222167, 0.195722, 
                         0.210882, 0.198651, 0.204099, 0.205225, 0.216540, 
                         0.215737, 0.215030, 0.216897, 0.216760])

# Collect all per-example layer scores
all_examples = []

for prompt_data in data:
    for seed_data in prompt_data['seeds']:
        # Extract scores for this example (one score per layer)
        example_scores = []
        for layer_data in sorted(seed_data['layers'], key=lambda x: x['layer']):
            example_scores.append(layer_data['score'])
        
        all_examples.append(example_scores)

# Convert to tensor: [n_examples, n_layers]
all_examples = torch.tensor(all_examples)

print(f"Shape: {all_examples.shape}")
print(f"Number of examples: {all_examples.shape[0]}")
print(f"Number of layers: {all_examples.shape[1]}")
print()

# Compute correlation for each example
pearson_correlations = []
spearman_correlations = []
pearson_pvalues = []
spearman_pvalues = []

for i, example_scores in enumerate(all_examples):
    # Pearson
    p_corr, p_pval = pearsonr(example_scores.numpy(), baseline.numpy())
    pearson_correlations.append(p_corr)
    pearson_pvalues.append(p_pval)
    
    # Spearman
    s_corr, s_pval = spearmanr(example_scores.numpy(), baseline.numpy())
    spearman_correlations.append(s_corr)
    spearman_pvalues.append(s_pval)

# Convert to arrays
pearson_correlations = np.array(pearson_correlations)
spearman_correlations = np.array(spearman_correlations)
pearson_pvalues = np.array(pearson_pvalues)
spearman_pvalues = np.array(spearman_pvalues)

# Analyze consistency
print("="*60)
print("CONSISTENCY ANALYSIS")
print("="*60)
print("\nPEARSON CORRELATION:")
print(f"  Mean:   {pearson_correlations.mean():.6f}")
print(f"  Std:    {pearson_correlations.std():.6f}")
print(f"  Min:    {pearson_correlations.min():.6f}")
print(f"  Max:    {pearson_correlations.max():.6f}")
print(f"  Median: {np.median(pearson_correlations):.6f}")
print(f"  Significant (p<0.05): {(pearson_pvalues < 0.05).sum()}/{len(pearson_pvalues)}")

print("\nSPEARMAN CORRELATION:")
print(f"  Mean:   {spearman_correlations.mean():.6f}")
print(f"  Std:    {spearman_correlations.std():.6f}")
print(f"  Min:    {spearman_correlations.min():.6f}")
print(f"  Max:    {spearman_correlations.max():.6f}")
print(f"  Median: {np.median(spearman_correlations):.6f}")
print(f"  Significant (p<0.05): {(spearman_pvalues < 0.05).sum()}/{len(spearman_pvalues)}")

print("\n" + "="*60)
print("DISTRIBUTION OF CORRELATIONS")
print("="*60)
print("\nPearson correlations by example:")
for i, (corr, pval) in enumerate(zip(pearson_correlations, pearson_pvalues)):
    sig = "*" if pval < 0.05 else " "
    print(f"  Example {i:3d}: {corr:7.4f} (p={pval:.4f}) {sig}")

print("\nSpearman correlations by example:")
for i, (corr, pval) in enumerate(zip(spearman_correlations, spearman_pvalues)):
    sig = "*" if pval < 0.05 else " "
    print(f"  Example {i:3d}: {corr:7.4f} (p={pval:.4f}) {sig}")

# Optional: Check if correlations are consistently positive/negative
print("\n" + "="*60)
print("SIGN CONSISTENCY")
print("="*60)
print(f"Pearson:  {(pearson_correlations > 0).sum()} positive, {(pearson_correlations < 0).sum()} negative")
print(f"Spearman: {(spearman_correlations > 0).sum()} positive, {(spearman_correlations < 0).sum()} negative")

Shape: torch.Size([50, 19])
Number of examples: 50
Number of layers: 19

CONSISTENCY ANALYSIS

PEARSON CORRELATION:
  Mean:   0.231793
  Std:    0.284742
  Min:    -0.381282
  Max:    0.759836
  Median: 0.262492
  Significant (p<0.05): 14/50

SPEARMAN CORRELATION:
  Mean:   0.129125
  Std:    0.234210
  Min:    -0.324039
  Max:    0.704017
  Median: 0.114827
  Significant (p<0.05): 3/50

DISTRIBUTION OF CORRELATIONS

Pearson correlations by example:
  Example   0: -0.0497 (p=0.8400)  
  Example   1: -0.1642 (p=0.5016)  
  Example   2:  0.2865 (p=0.2344)  
  Example   3:  0.1091 (p=0.6565)  
  Example   4: -0.0525 (p=0.8310)  
  Example   5:  0.3164 (p=0.1869)  
  Example   6:  0.5607 (p=0.0125) *
  Example   7:  0.5163 (p=0.0236) *
  Example   8:  0.4834 (p=0.0360) *
  Example   9:  0.4906 (p=0.0330) *
  Example  10:  0.5543 (p=0.0138) *
  Example  11:  0.4470 (p=0.0550)  
  Example  12:  0.5799 (p=0.0093) *
  Example  13:  0.2904 (p=0.2278)  
  Example  14:  0.2385 (p=0.3255)  
  Exam